In [4]:
from datasets import load_dataset

# Carga el dataset de Hugging Face
dataset = load_dataset("SINAI/ALIA-es-discriminative-stance-detection")

# Inspeccionar las particiones (train, validation, test)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'target', 'description', 'comment', 'annotation_1', 'annotation_2', 'annotation_3', 'majority_label'],
        num_rows: 3000
    })
})


In [5]:
import pandas as pd

# Convertimos el dataset a DataFrame
df = dataset["train"].to_pandas()

# Cantidad de comentarios por clase
print(df["majority_label"].value_counts())

majority_label
favor      1780
against     627
neutral     443
            150
Name: count, dtype: int64


In [6]:
# Comentarios sin etiqueta mayoritaria
sin_etiqueta = df[df["majority_label"] == ""]

print("Cantidad:", len(sin_etiqueta))

# Ver las anotaciones de los primeros 10
print(sin_etiqueta[
    ["annotation_1", "annotation_2", "annotation_3"]
].head(10))

Cantidad: 150
    annotation_1 annotation_2 annotation_3
2        against      neutral        favor
27       neutral        favor      against
29         favor      neutral      against
45       against        favor      neutral
58       against      neutral        favor
102        favor      against      neutral
111        favor      neutral      against
154      neutral      against        favor
161      against        favor      neutral
166        favor      against      neutral


In [7]:
# Verificar si los 150 casos tienen tres anotaciones distintas
sin_etiqueta[
    ["annotation_1", "annotation_2", "annotation_3"]
].nunique(axis=1).value_counts()

3    150
Name: count, dtype: int64

In [8]:
# Excluir comentarios sin etiqueta mayoritaria
df_limpio = df[df["majority_label"] != ""].copy()

# Verificar cantidad de comentarios
print("Comentarios originales:", len(df))
print("Comentarios utilizables:", len(df_limpio))

# Ver distribución de clases
print(df_limpio["majority_label"].value_counts())

Comentarios originales: 3000
Comentarios utilizables: 2850
majority_label
favor      1780
against     627
neutral     443
Name: count, dtype: int64


In [9]:
# Contar cuántas etiquetas distintas hay por comentario
acuerdos = df[
    ["annotation_1", "annotation_2", "annotation_3"]
].nunique(axis=1)

# Calcular tipos de acuerdo
acuerdo_total = (acuerdos == 1).sum()
acuerdo_parcial = (acuerdos == 2).sum()
sin_acuerdo = (acuerdos == 3).sum()

print("Acuerdo total:", acuerdo_total)
print("Acuerdo parcial:", acuerdo_parcial)
print("Sin acuerdo:", sin_acuerdo)

# Porcentajes
print("\nPorcentajes:")
print("Acuerdo total:", round(acuerdo_total / len(df) * 100, 2), "%")
print("Acuerdo parcial:", round(acuerdo_parcial / len(df) * 100, 2), "%")
print("Sin acuerdo:", round(sin_acuerdo / len(df) * 100, 2), "%")

Acuerdo total: 1539
Acuerdo parcial: 1311
Sin acuerdo: 150

Porcentajes:
Acuerdo total: 51.3 %
Acuerdo parcial: 43.7 %
Sin acuerdo: 5.0 %


In [10]:
# Calcular cantidad de palabras por comentario
df_limpio["num_palabras"] = df_limpio["comment"].str.split().str.len()

# Mostrar estadísticas
print(df_limpio["num_palabras"].describe())

count    2850.000000
mean       54.138596
std        32.073506
min         7.000000
25%        28.000000
50%        46.000000
75%        76.000000
max       155.000000
Name: num_palabras, dtype: float64


In [11]:
from sklearn.model_selection import train_test_split

# Primera división: 70% entrenamiento, 30% restante
train_df, temp_df = train_test_split(
    df_limpio,
    test_size=0.30,
    random_state=42,
    stratify=df_limpio["majority_label"]
)

# Segunda división: mitad validación, mitad test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["majority_label"]
)

print("Entrenamiento:", len(train_df))
print("Validación:", len(val_df))
print("Test:", len(test_df))

Entrenamiento: 1995
Validación: 427
Test: 428


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score

# Crear el modelo
modelo = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clasificador", LogisticRegression(max_iter=1000))
])

# Entrenar con los 1995 comentarios
modelo.fit(
    train_df["comment"],
    train_df["majority_label"]
)

# Predecir los 427 comentarios de validación
predicciones = modelo.predict(val_df["comment"])

# Evaluar resultados
print("Macro-F1:", f1_score(
    val_df["majority_label"],
    predicciones,
    average="macro"
))

print("Accuracy:", accuracy_score(
    val_df["majority_label"],
    predicciones
))

Macro-F1: 0.2969769846909843
Accuracy: 0.6229508196721312


In [13]:
from sklearn.metrics import classification_report

print(classification_report(
    val_df["majority_label"],
    predicciones,
    zero_division=0
))

              precision    recall  f1-score   support

     against       0.28      0.05      0.09        94
       favor       0.64      0.97      0.77       267
     neutral       0.33      0.02      0.03        66

    accuracy                           0.62       427
   macro avg       0.42      0.35      0.30       427
weighted avg       0.51      0.62      0.51       427



In [14]:
# Modelo con pesos balanceados
modelo_balanceado = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Entrenar
modelo_balanceado.fit(
    train_df["comment"],
    train_df["majority_label"]
)

# Predecir en validación
pred_balanceadas = modelo_balanceado.predict(val_df["comment"])

# Evaluar
print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_balanceadas,
    average="macro"
))

print("\nResultados por clase:")
print(classification_report(
    val_df["majority_label"],
    pred_balanceadas,
    zero_division=0
))

Macro-F1: 0.4522187604433166

Resultados por clase:
              precision    recall  f1-score   support

     against       0.43      0.54      0.48        94
       favor       0.72      0.69      0.71       267
     neutral       0.19      0.15      0.17        66

    accuracy                           0.58       427
   macro avg       0.45      0.46      0.45       427
weighted avg       0.58      0.58      0.57       427



In [15]:
modelo_bigramas = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

modelo_bigramas.fit(
    train_df["comment"],
    train_df["majority_label"]
)

pred_bigramas = modelo_bigramas.predict(val_df["comment"])

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_bigramas,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_bigramas,
    zero_division=0
))

Macro-F1: 0.47769261473175256
              precision    recall  f1-score   support

     against       0.41      0.49      0.45        94
       favor       0.73      0.80      0.76       267
     neutral       0.42      0.15      0.22        66

    accuracy                           0.63       427
   macro avg       0.52      0.48      0.48       427
weighted avg       0.61      0.63      0.61       427



In [16]:
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

for c in [0.01, 0.1, 1, 10, 100]:

    modelo_prueba = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clasificador", LogisticRegression(
            C=c,
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    modelo_prueba.fit(train_df["comment"], train_df["majority_label"])

    pred = modelo_prueba.predict(val_df["comment"])
    
    f1 = f1_score(
        val_df["majority_label"],
        pred,
        average="macro"
    )

    print(f"C={c}: Macro-F1={f1:.4f}")

C=0.01: Macro-F1=0.4571
C=0.1: Macro-F1=0.4512
C=1: Macro-F1=0.4777
C=10: Macro-F1=0.4449
C=100: Macro-F1=0.4525


In [17]:
modelo_caracteres = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5)
    )),
    ("clasificador", LogisticRegression(
        C=1,
        max_iter=1000,
        class_weight="balanced"
    ))
])

modelo_caracteres.fit(
    train_df["comment"],
    train_df["majority_label"]
)

pred_caracteres = modelo_caracteres.predict(val_df["comment"])

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_caracteres,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_caracteres,
    zero_division=0
))

Macro-F1: 0.4357141316771258
              precision    recall  f1-score   support

     against       0.41      0.48      0.44        94
       favor       0.71      0.71      0.71       267
     neutral       0.18      0.14      0.16        66

    accuracy                           0.57       427
   macro avg       0.43      0.44      0.44       427
weighted avg       0.56      0.57      0.57       427



In [18]:
# Combinar tema y comentario
X_train = train_df["target"].fillna("") + " " + train_df["comment"].fillna("")
X_val = val_df["target"].fillna("") + " " + val_df["comment"].fillna("")

modelo_con_target = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clasificador", LogisticRegression(
        C=1,
        max_iter=1000,
        class_weight="balanced"
    ))
])

modelo_con_target.fit(X_train, train_df["majority_label"])

pred_target = modelo_con_target.predict(X_val)

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_target,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_target,
    zero_division=0
))

Macro-F1: 0.49239509559350697
              precision    recall  f1-score   support

     against       0.46      0.56      0.51        94
       favor       0.74      0.78      0.76       267
     neutral       0.32      0.15      0.21        66

    accuracy                           0.64       427
   macro avg       0.51      0.50      0.49       427
weighted avg       0.62      0.64      0.62       427



In [19]:
X_train_completo = (
    train_df["target"].fillna("") + " " +
    train_df["description"].fillna("") + " " +
    train_df["comment"].fillna("")
)

X_val_completo = (
    val_df["target"].fillna("") + " " +
    val_df["description"].fillna("") + " " +
    val_df["comment"].fillna("")
)

modelo_completo = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clasificador", LogisticRegression(
        C=1,
        max_iter=1000,
        class_weight="balanced"
    ))
])

modelo_completo.fit(
    X_train_completo,
    train_df["majority_label"]
)

pred_completo = modelo_completo.predict(X_val_completo)

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_completo,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_completo,
    zero_division=0
))

Macro-F1: 0.4714662819501529
              precision    recall  f1-score   support

     against       0.43      0.53      0.48        94
       favor       0.74      0.63      0.68       267
     neutral       0.23      0.29      0.26        66

    accuracy                           0.56       427
   macro avg       0.47      0.48      0.47       427
weighted avg       0.59      0.56      0.57       427



In [20]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report

modelo_svm = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clasificador", LinearSVC(
        C=1,
        class_weight="balanced",
        random_state=42
    ))
])

modelo_svm.fit(X_train, train_df["majority_label"])

pred_svm = modelo_svm.predict(X_val)

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_svm,
    average="macro"
))

print(classification_report(
    val_df["majority_label"],
    pred_svm,
    zero_division=0
))

Macro-F1: 0.4613757353316921
              precision    recall  f1-score   support

     against       0.51      0.45      0.48        94
       favor       0.72      0.88      0.79       267
     neutral       0.26      0.08      0.12        66

    accuracy                           0.66       427
   macro avg       0.50      0.47      0.46       427
weighted avg       0.60      0.66      0.62       427



In [21]:
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score

configuraciones = ParameterGrid({
    "min_df": [1, 2, 3, 5],
    "sublinear_tf": [True, False],
    "max_features": [None, 5000, 10000]
})

resultados = []

for config in configuraciones:
    modelo_busqueda = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            **config
        )),
        ("clasificador", LogisticRegression(
            C=1,
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    modelo_busqueda.fit(X_train, train_df["majority_label"])
    pred = modelo_busqueda.predict(X_val)

    f1 = f1_score(
        val_df["majority_label"],
        pred,
        average="macro"
    )

    resultados.append({
        **config,
        "Macro-F1": f1
    })

resultados_df = pd.DataFrame(resultados)

display(
    resultados_df.sort_values(
        "Macro-F1",
        ascending=False
    ).head(10)
)

,max_features,min_df,sublinear_tf,Macro-F1
3,NaN,2,False,0.517155
5,NaN,3,False,0.507369
21,10000.0,3,False,0.507369
19,10000.0,2,False,0.504509
17,10000.0,1,False,0.504429
4,NaN,3,True,0.500488
20,10000.0,3,True,0.500184
2,NaN,2,True,0.498684
18,10000.0,2,True,0.495974
11,5000.0,2,False,0.495871


In [22]:
import time
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

modelo_optimizado = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=False,
        max_features=None
    )),
    ("clasificador", LogisticRegression(
        C=1,
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Entrenamiento
inicio = time.perf_counter()
modelo_optimizado.fit(X_train, train_df["majority_label"])
tiempo_entrenamiento_opt = time.perf_counter() - inicio

# Inferencia
inicio = time.perf_counter()
pred_optimizado = modelo_optimizado.predict(X_val)
tiempo_inferencia_opt = time.perf_counter() - inicio

# Métricas
y_real = val_df["majority_label"]

print("Accuracy:", accuracy_score(y_real, pred_optimizado))
print("Precision Macro:", precision_score(y_real, pred_optimizado, average="macro"))
print("Recall Macro:", recall_score(y_real, pred_optimizado, average="macro"))
print("Macro-F1:", f1_score(y_real, pred_optimizado, average="macro"))
print("Weighted-F1:", f1_score(y_real, pred_optimizado, average="weighted"))

print("\nReporte completo:")
print(classification_report(y_real, pred_optimizado, zero_division=0))

print("\nMatriz de confusión:")
print(confusion_matrix(
    y_real,
    pred_optimizado,
    labels=["against", "favor", "neutral"]
))

print("\nTiempos:")
print("Entrenamiento:", round(tiempo_entrenamiento_opt, 3), "segundos")
print("Inferencia:", round(tiempo_inferencia_opt, 3), "segundos")
print("Por comentario:", round(tiempo_inferencia_opt / len(X_val) * 1000, 3), "ms")

Accuracy: 0.6323185011709602
Precision Macro: 0.524874664767735
Recall Macro: 0.5188944187270746
Macro-F1: 0.5171550795585741
Weighted-F1: 0.6275807605900855

Reporte completo:
              precision    recall  f1-score   support

     against       0.45      0.54      0.49        94
       favor       0.75      0.76      0.76       267
     neutral       0.37      0.26      0.30        66

    accuracy                           0.63       427
   macro avg       0.52      0.52      0.52       427
weighted avg       0.63      0.63      0.63       427


Matriz de confusión:
[[ 51  33  10]
 [ 46 202  19]
 [ 16  33  17]]

Tiempos:
Entrenamiento: 0.392 segundos
Inferencia: 0.028 segundos
Por comentario: 0.066 ms


In [23]:
resumen_modelos = pd.DataFrame([
    ["Baseline", 0.2970],
    ["Balanceado", 0.4522],
    ["Bigramas", 0.4777],
    ["Bigramas + target", 0.4924],
    ["SVM", 0.4614],
    ["TF-IDF optimizado", 0.5172]
], columns=["Modelo", "Macro-F1 validación"])

display(resumen_modelos)

,Modelo,Macro-F1 validación
0,Baseline,0.2970
1,Balanceado,0.4522
2,Bigramas,0.4777
3,Bigramas + target,0.4924
4,SVM,0.4614
5,TF-IDF optimizado,0.5172


In [24]:
import joblib

joblib.dump(modelo_optimizado, "modelo_clasico.pkl")

['modelo_clasico.pkl']

## BETO sin fine-tuning

In [25]:
%pip install "numpy<2" "torch==2.2.2" "transformers==4.44.2" "accelerate==0.34.2"

Note: you may need to restart the kernel to use updated packages.


In [26]:
%pip install "pandas==2.2.3" "datasets==3.6.0"

Note: you may need to restart the kernel to use updated packages.


In [27]:
%pip install "pyarrow==17.0.0"

Note: you may need to restart the kernel to use updated packages.


In [28]:
import torch
import transformers
import datasets
import pandas
import numpy

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("GPU disponible:", torch.cuda.is_available())

PyTorch: 2.2.2
Transformers: 4.44.2
Datasets: 3.6.0
Pandas: 2.2.3
NumPy: 1.26.4
GPU disponible: False


In [29]:
from transformers import AutoTokenizer, AutoModel
import torch

nombre_modelo = "dccuchile/bert-base-spanish-wwm-cased"

# Cargar tokenizador y modelo preentrenado
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
beto = AutoModel.from_pretrained(nombre_modelo)

# Usar CPU y modo evaluación
device = torch.device("cpu")
beto.to(device)
beto.eval()

print("BETO cargado correctamente")

/Users/paulaho/opt/anaconda3/envs/tp3_nlp/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BETO cargado correctamente


In [30]:
textos_train = (
    train_df["target"].fillna("").astype(str)
    + " [SEP] " +
    train_df["comment"].fillna("").astype(str)
).tolist()

textos_val = (
    val_df["target"].fillna("").astype(str)
    + " [SEP] " +
    val_df["comment"].fillna("").astype(str)
).tolist()

print("Entrenamiento:", len(textos_train))
print("Validación:", len(textos_val))
print("Ejemplo:", textos_train[0][:200])

Entrenamiento: 1995
Validación: 427
Ejemplo: Ruido nocturno de bares, discotecas y gente en la calle [SEP]  Vivo en un barrio donde el botellón es indiscriminado, zona de Guzmán el Bueno-General Rodrigo- Julián Romea-General Dávila, todas las no


In [31]:
import numpy as np
from tqdm.auto import tqdm
import time

def extraer_embeddings(textos, batch_size=8):
    embeddings = []
    inicio = time.perf_counter()

    for i in tqdm(range(0, len(textos), batch_size)):
        lote = textos[i:i + batch_size]

        entradas = tokenizer(
            lote,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            salida = beto(**entradas)

        # Promedio de los tokens, ignorando el padding
        mascara = entradas["attention_mask"].unsqueeze(-1)
        vectores = (
            (salida.last_hidden_state * mascara).sum(dim=1)
            / mascara.sum(dim=1)
        )

        embeddings.append(vectores.cpu().numpy())

    tiempo = time.perf_counter() - inicio
    return np.vstack(embeddings), tiempo


emb_train, tiempo_emb_train = extraer_embeddings(textos_train)
emb_val, tiempo_emb_val = extraer_embeddings(textos_val)

print("Embeddings entrenamiento:", emb_train.shape)
print("Embeddings validación:", emb_val.shape)
print("Tiempo entrenamiento:", round(tiempo_emb_train, 2), "segundos")
print("Tiempo validación:", round(tiempo_emb_val, 2), "segundos")

100%|██████████| 54/54 [02:31<00:00,  2.81s/it]

Embeddings entrenamiento: (1995, 768)
Embeddings validación: (427, 768)
Tiempo entrenamiento: 447.06 segundos
Tiempo validación: 151.89 segundos


In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report
import time

# Entrenar clasificador con embeddings de BETO
clasificador_beto = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

inicio = time.perf_counter()

clasificador_beto.fit(
    emb_train,
    train_df["majority_label"]
)

tiempo_clasificador = time.perf_counter() - inicio

# Predecir en validación
pred_beto = clasificador_beto.predict(emb_val)

print("Macro-F1:", f1_score(
    val_df["majority_label"],
    pred_beto,
    average="macro"
))

print("Accuracy:", accuracy_score(
    val_df["majority_label"],
    pred_beto
))

print("\nResultados por clase:")
print(classification_report(
    val_df["majority_label"],
    pred_beto,
    zero_division=0
))

print("Tiempo entrenamiento clasificador:", round(tiempo_clasificador, 2), "segundos")

Macro-F1: 0.45444638280459176
Accuracy: 0.522248243559719

Resultados por clase:
              precision    recall  f1-score   support

     against       0.38      0.51      0.44        94
       favor       0.75      0.57      0.65       267
     neutral       0.23      0.35      0.28        66

    accuracy                           0.52       427
   macro avg       0.46      0.48      0.45       427
weighted avg       0.59      0.52      0.54       427

Tiempo entrenamiento clasificador: 1.4 segundos


In [33]:
import numpy as np

np.save("emb_train_beto.npy", emb_train)
np.save("emb_val_beto.npy", emb_val)

print("Embeddings guardados correctamente")

Embeddings guardados correctamente


In [34]:
from transformers import AutoModelForSequenceClassification
import torch
import time

# Convertir las etiquetas a números
label2id = {
    "against": 0,
    "favor": 1,
    "neutral": 2
}

id2label = {v: k for k, v in label2id.items()}

# Crear BETO con una capa de clasificación
beto_ft = AutoModelForSequenceClassification.from_pretrained(
    "dccuchile/bert-base-spanish-wwm-cased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

beto_ft.to(device)

# Usamos solo 16 textos para probar la velocidad
textos_prueba = textos_train[:16]

etiquetas_prueba = torch.tensor(
    train_df["majority_label"].iloc[:16].map(label2id).tolist()
).to(device)

entradas_prueba = tokenizer(
    textos_prueba,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
).to(device)

print("BETO preparado para fine-tuning")
print("Textos de prueba:", len(textos_prueba))
print("Etiquetas:", etiquetas_prueba.shape)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BETO preparado para fine-tuning
Textos de prueba: 16
Etiquetas: torch.Size([16])


In [35]:
# Preparar el optimizador
optimizer = torch.optim.AdamW(
    beto_ft.parameters(),
    lr=2e-5
)

# Activar modo entrenamiento
beto_ft.train()

inicio = time.perf_counter()

# Una actualización de entrenamiento
optimizer.zero_grad()

salida = beto_ft(
    **entradas_prueba,
    labels=etiquetas_prueba
)

loss = salida.loss

loss.backward()
optimizer.step()

tiempo = time.perf_counter() - inicio

print("Loss:", round(loss.item(), 4))
print("Tiempo para 16 comentarios:", round(tiempo, 2), "segundos")

# Estimación aproximada para una época completa
pasos_epoca = (len(textos_train) + 15) // 16

print(
    "Tiempo estimado por época:",
    round(tiempo * pasos_epoca / 60, 1),
    "minutos"
)

Loss: 1.1013
Tiempo para 16 comentarios: 13.89 segundos
Tiempo estimado por época: 28.9 minutos


In [36]:
from torch.utils.data import TensorDataset, DataLoader

# Cargar BETO nuevamente desde cero
beto_ft = AutoModelForSequenceClassification.from_pretrained(
    nombre_modelo,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
).to(device)

# Tokenizar todos los textos de entrenamiento
tokens_train = tokenizer(
    textos_train,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Etiquetas numéricas
y_train = torch.tensor(
    train_df["majority_label"].map(label2id).tolist()
)

# Crear dataset
dataset_train = TensorDataset(
    tokens_train["input_ids"],
    tokens_train["attention_mask"],
    y_train
)

# Crear lotes de 16 comentarios
loader_train = DataLoader(
    dataset_train,
    batch_size=16,
    shuffle=True
)

print("Comentarios:", len(dataset_train))
print("Lotes por época:", len(loader_train))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Comentarios: 1995
Lotes por época: 125


In [37]:
from tqdm.auto import tqdm
import time

optimizer = torch.optim.AdamW(
    beto_ft.parameters(),
    lr=2e-5
)

beto_ft.train()
inicio = time.perf_counter()
perdida_total = 0

for input_ids, attention_mask, labels in tqdm(loader_train):

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    salida = beto_ft(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = salida.loss
    loss.backward()
    optimizer.step()

    perdida_total += loss.item()

tiempo_entrenamiento = time.perf_counter() - inicio

print("Loss promedio:", round(perdida_total / len(loader_train), 4))
print("Tiempo de entrenamiento:", round(tiempo_entrenamiento / 60, 2), "minutos")

# Guardar el modelo entrenado
beto_ft.save_pretrained("beto_finetuned")
tokenizer.save_pretrained("beto_finetuned")

print("Modelo guardado correctamente")

100%|██████████| 125/125 [32:35<00:00, 15.64s/it]


Loss promedio: 0.8492
Tiempo de entrenamiento: 32.59 minutos
Modelo guardado correctamente


In [38]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
import numpy as np
import time

beto_ft.eval()

predicciones = []
inicio = time.perf_counter()

for i in range(0, len(textos_val), 16):
    lote = textos_val[i:i+16]

    entradas = tokenizer(
        lote,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        salida = beto_ft(**entradas)

    pred = torch.argmax(salida.logits, dim=1)
    predicciones.extend(pred.cpu().numpy())

tiempo_inferencia = time.perf_counter() - inicio

pred_labels = [id2label[int(p)] for p in predicciones]
y_real = val_df["majority_label"].tolist()

print("Macro-F1:", f1_score(y_real, pred_labels, average="macro"))
print("Accuracy:", accuracy_score(y_real, pred_labels))
print("\n", classification_report(y_real, pred_labels, zero_division=0))
print("Tiempo de inferencia:", round(tiempo_inferencia, 2), "segundos")

Macro-F1: 0.4526056089877004
Accuracy: 0.6768149882903981

               precision    recall  f1-score   support

     against       0.61      0.27      0.37        94
       favor       0.68      0.96      0.80       267
     neutral       1.00      0.11      0.19        66

    accuracy                           0.68       427
   macro avg       0.76      0.44      0.45       427
weighted avg       0.71      0.68      0.61       427

Tiempo de inferencia: 80.97 segundos


In [39]:
resultados_beto_ft = [{
    "epocas": 1,
    "learning_rate": 2e-5,
    "macro_f1": 0.4526056089877004,
    "accuracy": 0.6768149882903981,
    "tiempo_entrenamiento_seg": tiempo_entrenamiento,
    "tiempo_inferencia_seg": tiempo_inferencia
}]

print("Resultados de la primera época guardados")

Resultados de la primera época guardados


In [40]:
# Continuar entrenando el mismo modelo
beto_ft.train()

inicio = time.perf_counter()
perdida_total = 0

for input_ids, attention_mask, labels in tqdm(loader_train):

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    salida = beto_ft(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = salida.loss
    loss.backward()
    optimizer.step()

    perdida_total += loss.item()

tiempo_epoca_2 = time.perf_counter() - inicio

print("Loss promedio época 2:", round(perdida_total / len(loader_train), 4))
print("Tiempo época 2:", round(tiempo_epoca_2 / 60, 2), "minutos")

# Guardar modelo de 2 épocas en otra carpeta
beto_ft.save_pretrained("beto_finetuned_2ep")
tokenizer.save_pretrained("beto_finetuned_2ep")

print("Modelo de 2 épocas guardado correctamente")

100%|██████████| 125/125 [30:54<00:00, 14.84s/it]


Loss promedio época 2: 0.5956
Tiempo época 2: 30.92 minutos
Modelo de 2 épocas guardado correctamente


In [41]:
# Evaluar BETO después de 2 épocas
beto_ft.eval()

predicciones_2ep = []
inicio = time.perf_counter()

for i in range(0, len(textos_val), 16):
    lote = textos_val[i:i+16]

    entradas = tokenizer(
        lote,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        salida = beto_ft(**entradas)

    pred = torch.argmax(salida.logits, dim=1)
    predicciones_2ep.extend(pred.cpu().numpy())

tiempo_inferencia_2ep = time.perf_counter() - inicio

pred_labels_2ep = [id2label[int(p)] for p in predicciones_2ep]
y_real = val_df["majority_label"].tolist()

macro_f1_2ep = f1_score(y_real, pred_labels_2ep, average="macro")
accuracy_2ep = accuracy_score(y_real, pred_labels_2ep)

print("Macro-F1:", macro_f1_2ep)
print("Accuracy:", accuracy_2ep)
print(classification_report(y_real, pred_labels_2ep, zero_division=0))
print("Tiempo de inferencia:", round(tiempo_inferencia_2ep, 2), "segundos")

# Registrar resultados
resultados_beto_ft.append({
    "epocas": 2,
    "learning_rate": 2e-5,
    "macro_f1": macro_f1_2ep,
    "accuracy": accuracy_2ep,
    "tiempo_entrenamiento_seg": tiempo_entrenamiento + tiempo_epoca_2,
    "tiempo_inferencia_seg": tiempo_inferencia_2ep
})

Macro-F1: 0.5028119623653736
Accuracy: 0.6955503512880562
              precision    recall  f1-score   support

     against       0.65      0.44      0.52        94
       favor       0.70      0.93      0.80       267
     neutral       0.70      0.11      0.18        66

    accuracy                           0.70       427
   macro avg       0.68      0.49      0.50       427
weighted avg       0.69      0.70      0.64       427

Tiempo de inferencia: 72.11 segundos


In [42]:
import pandas as pd

df_resultados_beto = pd.DataFrame(resultados_beto_ft)

df_resultados_beto.to_csv(
    "resultados_beto_finetuning.csv",
    index=False
)

print(df_resultados_beto)
print("Resultados guardados correctamente")

   epocas  learning_rate  macro_f1  accuracy  tiempo_entrenamiento_seg  \
0       1        0.00002  0.452606  0.676815               1955.443833   
1       2        0.00002  0.502812  0.695550               3810.388267   

   tiempo_inferencia_seg  
0              80.967704  
1              72.113646  
Resultados guardados correctamente


In [43]:
# Nuevo modelo para experimentar con otro learning rate
beto_ft_lr3 = AutoModelForSequenceClassification.from_pretrained(
    nombre_modelo,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
).to(device)

# Nuevo optimizador
optimizer_lr3 = torch.optim.AdamW(
    beto_ft_lr3.parameters(),
    lr=3e-5
)

print("Nuevo BETO preparado")
print("Learning rate:", optimizer_lr3.param_groups[0]["lr"])
print("Épocas previstas: 1")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Nuevo BETO preparado
Learning rate: 3e-05
Épocas previstas: 1


In [44]:
beto_ft_lr3.train()

inicio = time.perf_counter()
perdida_total_lr3 = 0

for input_ids, attention_mask, labels in tqdm(loader_train):

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    labels = labels.to(device)

    optimizer_lr3.zero_grad()

    salida = beto_ft_lr3(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = salida.loss
    loss.backward()
    optimizer_lr3.step()

    perdida_total_lr3 += loss.item()

tiempo_lr3 = time.perf_counter() - inicio

print("Loss promedio:", round(perdida_total_lr3 / len(loader_train), 4))
print("Tiempo entrenamiento:", round(tiempo_lr3 / 60, 2), "minutos")

# Guardar modelo
beto_ft_lr3.save_pretrained("beto_finetuned_lr3")
tokenizer.save_pretrained("beto_finetuned_lr3")

print("Modelo guardado correctamente")

100%|██████████| 125/125 [20:19<00:00,  9.76s/it]


Loss promedio: 0.8672
Tiempo entrenamiento: 20.33 minutos
Modelo guardado correctamente


In [45]:
beto_ft_lr3.eval()

predicciones_lr3 = []
inicio = time.perf_counter()

for i in range(0, len(textos_val), 16):
    lote = textos_val[i:i+16]

    entradas = tokenizer(
        lote,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        salida = beto_ft_lr3(**entradas)

    pred = torch.argmax(salida.logits, dim=1)
    predicciones_lr3.extend(pred.cpu().numpy())

tiempo_inferencia_lr3 = time.perf_counter() - inicio

pred_labels_lr3 = [id2label[int(p)] for p in predicciones_lr3]
y_real = val_df["majority_label"].tolist()

macro_f1_lr3 = f1_score(y_real, pred_labels_lr3, average="macro")
accuracy_lr3 = accuracy_score(y_real, pred_labels_lr3)

print("Macro-F1:", macro_f1_lr3)
print("Accuracy:", accuracy_lr3)
print(classification_report(y_real, pred_labels_lr3, zero_division=0))
print("Tiempo inferencia:", round(tiempo_inferencia_lr3, 2), "segundos")

resultados_beto_ft.append({
    "epocas": 1,
    "learning_rate": 3e-5,
    "macro_f1": macro_f1_lr3,
    "accuracy": accuracy_lr3,
    "tiempo_entrenamiento_seg": tiempo_lr3,
    "tiempo_inferencia_seg": tiempo_inferencia_lr3
})

pd.DataFrame(resultados_beto_ft).to_csv(
    "resultados_beto_finetuning.csv",
    index=False
)

Macro-F1: 0.4655177606119469
Accuracy: 0.6604215456674473
              precision    recall  f1-score   support

     against       0.58      0.23      0.33        94
       favor       0.68      0.93      0.79       267
     neutral       0.46      0.20      0.28        66

    accuracy                           0.66       427
   macro avg       0.58      0.45      0.47       427
weighted avg       0.63      0.66      0.61       427

Tiempo inferencia: 74.33 segundos


In [46]:
beto_ft_lr3.eval()

predicciones_lr3 = []
inicio = time.perf_counter()

for i in range(0, len(textos_val), 16):
    lote = textos_val[i:i+16]

    entradas = tokenizer(
        lote,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        salida = beto_ft_lr3(**entradas)

    pred = torch.argmax(salida.logits, dim=1)
    predicciones_lr3.extend(pred.cpu().numpy())

tiempo_inferencia_lr3 = time.perf_counter() - inicio

pred_labels_lr3 = [id2label[int(p)] for p in predicciones_lr3]
y_real = val_df["majority_label"].tolist()

macro_f1_lr3 = f1_score(y_real, pred_labels_lr3, average="macro")
accuracy_lr3 = accuracy_score(y_real, pred_labels_lr3)

print("Macro-F1:", macro_f1_lr3)
print("Accuracy:", accuracy_lr3)
print(classification_report(y_real, pred_labels_lr3, zero_division=0))
print("Tiempo inferencia:", round(tiempo_inferencia_lr3, 2), "segundos")

resultados_beto_ft.append({
    "epocas": 1,
    "learning_rate": 3e-5,
    "macro_f1": macro_f1_lr3,
    "accuracy": accuracy_lr3,
    "tiempo_entrenamiento_seg": tiempo_lr3,
    "tiempo_inferencia_seg": tiempo_inferencia_lr3
})

pd.DataFrame(resultados_beto_ft).to_csv(
    "resultados_beto_finetuning.csv",
    index=False
)

Macro-F1: 0.4655177606119469
Accuracy: 0.6604215456674473
              precision    recall  f1-score   support

     against       0.58      0.23      0.33        94
       favor       0.68      0.93      0.79       267
     neutral       0.46      0.20      0.28        66

    accuracy                           0.66       427
   macro avg       0.58      0.45      0.47       427
weighted avg       0.63      0.66      0.61       427

Tiempo inferencia: 91.49 segundos
